# Musicm8 — Complete Song + Neural Vocals

This is the main one-click workflow. It now **must clearly report whether singing was actually created** instead of silently presenting an instrumental as a final song.

The vocal backend first tries ACE-Step 1.5 **Lego vocals** over Musicm8's backing. On a Colab T4 it automatically enables CPU offload. If Lego fails, Musicm8 automatically tries a second route: **ACE-Step cover generation → Demucs vocal extraction → Musicm8 vocal FX/mix**.

Change `IDEA` and `VOCAL_STYLE`, then run the big cell. The first vocal run can be large because ACE-Step and the base model are cached in Drive.

In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK COMPLETE SONG + REAL VOCAL STATUS
# ============================================================

import os, sys, json, shutil, subprocess
from pathlib import Path

# ------------------------------------------------------------
# EDIT THESE
# ------------------------------------------------------------
IDEA = "dark UK garage song about knowing a relationship is over but not being able to leave, emotional chords, deep moving bass"
BARS = 32
SEED = 42

VOCALS = True
VOCAL_STYLE = "expressive contemporary lead vocal, intimate verses, emotional hook, clear lyrics, modern UK electronic production"
VOCAL_LANGUAGE = "en"
VOCAL_STEPS = 32

MATCH_ITERS = 48
MATCH_SECONDS = 3.0
FORCE_SOUND_MATCH = False
AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO = ROOT / "audio"
WORK = ROOT / "work"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
AUDIO.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(WORK / "hf_cache")
os.environ["UV_CACHE_DIR"] = str(WORK / "uv_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Always refresh Musicm8 without throwing away the current T4 session.
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
print("Installing Musicm8 dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ai.txt"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng"], check=False)

import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU connected. Runtime → Change runtime type → GPU, then run THIS SAME CELL again.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

audio_files = [p for p in AUDIO.rglob("*") if p.is_file() and p.suffix.lower() in {".wav",".mp3",".flac",".m4a",".aac",".ogg",".opus"}]
print("Reference songs:", len(audio_files))
if not audio_files:
    raise FileNotFoundError(f"Put your reference songs in {AUDIO}")

cmd = [
    sys.executable, "-u", "ai_producer_workflow.py",
    "--root", str(ROOT),
    "--repo", str(REPO),
    "--idea", IDEA,
    "--bars", str(BARS),
    "--seed", str(SEED),
    "--ai-model", AI_MODEL,
    "--match-iters", str(MATCH_ITERS),
    "--match-seconds", str(MATCH_SECONDS),
    "--vocal-style", VOCAL_STYLE,
    "--vocal-language", VOCAL_LANGUAGE,
    "--vocal-steps", str(VOCAL_STEPS),
]
if not VOCALS:
    cmd.append("--no-vocals")
if FORCE_SOUND_MATCH:
    cmd.append("--force-sound-match")

print("\n🚀 Starting Musicm8 complete-song workflow...")
subprocess.run(cmd, check=True)

PROJECT = WORK / "ai_projects/latest"
MASTER = PROJECT / "master.wav"
INSTRUMENTAL = PROJECT / "master_instrumental.wav"
LYRICS = PROJECT / "lyrics.txt"
RAW_VOCAL = PROJECT / "vocals/neural_lead_raw.wav"
VOCAL = PROJECT / "vocals/vocal_mix.wav"
VOCAL_STATUS = PROJECT / "vocals/vocal_status.json"
VOCAL_LOG = PROJECT / "vocals/vocal_backend.log"

print("\n============================================================")
print("MUSICM8 RESULT")
print("============================================================")
print("Lyrics      :", LYRICS)
print("Vocal score :", PROJECT / "vocal_score.json")
print("MIDI        :", PROJECT / "arrangement.mid")
print("Instruments :", PROJECT / "audio_stems_polished")
print("Final master:", MASTER)

status = {}
if VOCAL_STATUS.exists():
    status = json.loads(VOCAL_STATUS.read_text(encoding="utf-8"))

if not VOCALS:
    print("\n🎹 VOCALS DISABLED — instrumental requested")
elif RAW_VOCAL.exists() and VOCAL.exists():
    method = status.get("method", "ACE-Step")
    print(f"\n✅ SINGING CREATED — method: {method}")
    print("Raw vocal   :", RAW_VOCAL)
    print("Vocal mix   :", VOCAL)
    if status.get("rms_db") is not None:
        print("Vocal RMS   :", status.get("rms_db"), "dB")
else:
    print("\n❌ NO SINGING WAS CREATED")
    print("The file below is therefore an INSTRUMENTAL FALLBACK, not a completed vocal song.")
    if status.get("errors"):
        print("Vocal backend errors:")
        for err in status["errors"]:
            print(" -", err)
    if VOCAL_LOG.exists():
        lines = VOCAL_LOG.read_text(encoding="utf-8", errors="ignore").splitlines()
        print("\nLast vocal-backend log lines:")
        print("\n".join(lines[-35:]))

if LYRICS.exists():
    print("\n📝 LYRICS\n")
    print(LYRICS.read_text(encoding="utf-8"))

from IPython.display import Audio, display
if RAW_VOCAL.exists():
    print("\n🎤 RAW NEURAL VOCAL")
    display(Audio(str(RAW_VOCAL)))
if VOCAL.exists():
    print("\n🎚️ PROCESSED VOCAL MIX")
    display(Audio(str(VOCAL)))
if INSTRUMENTAL.exists():
    print("\n🎹 INSTRUMENTAL")
    display(Audio(str(INSTRUMENTAL)))
if MASTER.exists():
    print("\n🎵 MUSICM8 FINAL SONG" if VOCAL.exists() else "\n🎵 MUSICM8 INSTRUMENTAL FALLBACK")
    display(Audio(str(MASTER)))


## Vocal diagnostics

If singing still cannot be created, this cell shows the saved status and exact backend log.

In [ ]:
from pathlib import Path
import json
project = Path('/content/drive/MyDrive/Musicm8/work/ai_projects/latest')
status = project / 'vocals/vocal_status.json'
log = project / 'vocals/vocal_backend.log'
print(status.read_text() if status.exists() else 'No vocal_status.json')
if log.exists():
    print('\n--- vocal backend log tail ---')
    print('\n'.join(log.read_text(errors='ignore').splitlines()[-80:]))
